In [ ]:
import numpy as np
import scipy.stats as stats
import scipy.special as special
import openturns as ot
from typing import Tuple

In [3]:
import sys
import os

# Set project root relative to notebook location
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [4]:
from Losses import gaussian_xgb_loss as gauss
from Losses import clayton_xgb_loss as clayton
from Losses import gumbel_xgb_loss as gumbel

In [ ]:
def sample_gaussian_copula(rho_array, seed):
    """
    Sample one bivariate observation from a Gaussian copula for each rho in rho_array.

    Parameters:
        rho_array (np.ndarray): shape (n,), correlation coefficients in (-1, 1)

    Returns:
        np.ndarray: shape (n, 2), samples from the Gaussian copula
    """
    ot.RandomGenerator.SetSeed(seed)
    n = len(rho_array)
    samples = np.zeros((n, 2))

    for i, rho in enumerate(rho_array):
        if not (-1 < rho < 1):
            raise ValueError("Each rho must be in the interval (-1, 1).")

        # Define correlation matrix
        corr_matrix = ot.CorrelationMatrix(2)
        corr_matrix[0, 1] = rho

        # Create the Gaussian copula
        copula = ot.NormalCopula(corr_matrix)

        # Draw a single sample
        u = copula.getSample(1)[0]
        samples[i] = [u[0], u[1]]

    samples = np.array(samples)
    # Clip to avoid 0 or 1
    eps = 1e-10
    samples = np.clip(samples, eps, 1 - eps)
    return samples

def sample_clayton_copula(theta_array, seed):
    """
    Sample one point from a bivariate Clayton copula for each parameter value in theta_array.
    
    Parameters:
        theta_array (np.ndarray): 1D array of positive Clayton copula parameters (theta > 0).
    
    Returns:
        samples (np.ndarray): Array of shape (n, 2) where each row is a sample (u1, u2).
    """
    ot.RandomGenerator.SetSeed(seed)
    samples = []
    for theta in theta_array:
        copula = ot.ClaytonCopula(theta)
        sample = copula.getSample(1)
        samples.append(sample[0])

    samples = np.array(samples)
    eps = 1e-10
    samples = np.clip(samples, eps, 1 - eps)
    return samples

def sample_gumbel_copula(theta_array, seed):
    """
    Sample one observation from a bivariate Gumbel copula for each theta.
    Uses OpenTURNS for sampling.
    
    Parameters:
        theta_array (np.ndarray): array of Gumbel copula parameters (theta >= 1)
    
    Returns:
        np.ndarray: shape (n, 2), copula samples in (0, 1)
    """
    ot.RandomGenerator.SetSeed(seed)
    samples = []
    for theta in theta_array:
        copula = ot.GumbelCopula(theta)
        sample = copula.getSample(1)
        samples.append(sample[0])
    
    samples = np.array(samples)
    # Clip to avoid 0 or 1
    eps = 1e-10
    samples = np.clip(samples, eps, 1 - eps)
    return samples

In [ ]:
# shape of generated data.
n = 2000
p = 10

ratio = 0.7
seed = 2000

num_boost_round = 500

np.random.seed(seed=seed)

x = np.random.uniform(0, 1, (n, p))
x1 = x[:, 0]
x2 = x[:, 1]
x3 = x[:, 2]

# First linear scenario
def true_tau1(X):
    x1 = X[:, 0]
    x4 = X[:,3]

    return special.expit(1.2*x1 - 1.5*x4)

tau1 = true_tau1(x)

    # Second non-linear scenario
def true_tau2(X):
    x1 = X[:, 0]
    x4 = X[:, 3]

    return special.expit(0.8*np.sin(2*np.pi*x1) + 1.5*(x1 > 0.5) - 1.2*(x4 > 0.7) + 0.6*x4**2)

tau2 = true_tau2(x)

def clip_copula_samples(samples, eps=1e-10):
    """
    Ensure copula samples are strictly inside (0, 1)^2.
    
    Parameters:
        samples (np.ndarray): shape (n, 2), values in [0, 1]
        eps (float): small buffer to keep samples strictly in (0, 1)
    
    Returns:
        np.ndarray: clipped samples in (eps, 1 - eps)
    """
    return np.clip(samples, eps, 1 - eps)

In [ ]:
def gaussian_lpdf(u, rho):
    """
    Gaussian copula log-likelihood evaluation

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Gaussian copula log-likelihood
    """
    a = stats.norm.ppf(u[:, 0]).reshape(-1,1)
    b = stats.norm.ppf(u[:, 1]).reshape(-1,1)
    c = a*b
    d = a**2 + b**2
    return np.sum(-np.log(np.sqrt(1-rho**2)) - ((rho**2)*d - 2*rho*c)/(2*(1-rho**2)))

def gaussian_lpdf_grad(u, rho):
    """
    Gaussian copula gradient w.r.t copula parameter rho

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Gaussian copula lpdf gradient
    """
    a = stats.norm.ppf(u[:, 0]).reshape(-1,1)
    b = stats.norm.ppf(u[:, 1]).reshape(-1,1)
    c = a*b
    d = a**2 + b**2
    return rho/(1-rho**2) + ((1+rho**2)*c - d*rho)/((1-rho**2)**2)
    
def gaussian_inv_link(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: inverse link function of transformed copula parameter
    """
    return (2/np.pi)*np.arctan(tpar)
    
def gaussian_inv_link_grad(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    return: gradient of inverse link function w.r.t transformed copula parameter
    """
    return (2/np.pi)*(1/(np.square(tpar)+1))
    
def gaussian_gradient(u: np.ndarray, predt: np.ndarray) -> np.ndarray:
    '''Compute the gradient of the nll'''
    rho = gaussian_inv_link(predt)
    return -gaussian_lpdf_grad(u, rho)*gaussian_inv_link_grad(predt)
        
def gaussian_copula_nll_obj(u: np.ndarray, predt: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''Gaussian copula negative log-likelihood objective'''
    grad = gaussian_gradient(u, predt)
    hess = grad**2 #BHHH hessian substitute
    return grad, hess

def gaussian_copula_nll_metric(u: np.ndarray, predt: np.ndarray) -> Tuple[str, float]:
    ''' Gaussian copula negative log-likelihood metric'''
    rho = gaussian_inv_link(predt)
    log_likelihood = gaussian_lpdf(u, rho)
    return float(-log_likelihood)

In [ ]:
def clayton_Afunc(u, theta):
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    a = np.hstack((-theta*np.log(u), np.zeros_like(u0)))
    b = np.hstack((np.ones_like(u0), np.ones_like(u1), -np.ones_like(u0)))

    return np.exp(special.logsumexp(a, b=b, axis=1)).reshape(-1,1) #u0**(-theta) + u1**(-theta) - 1

def clayton_logAfunc(u, theta):
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    a = np.hstack((-theta*np.log(u), np.zeros_like(u0)))
    b = np.hstack((np.ones_like(u0), np.ones_like(u1), -np.ones_like(u0)))

    return special.logsumexp(a, b=b, axis=1).reshape(-1,1)
    
def clayton_Afunc_grad(u, theta):
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    return -((u0**(-theta))*np.log(u0) + (u1**(-theta))*np.log(u1))

def clayton_logA_grad(u, theta):
    A_func = clayton_Afunc(u, theta)
    return clayton_Afunc_grad(u, theta)/A_func

def clayton_lpdf(u, theta):
    """
    Clayton copula log-likelihood evaluation

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Clayton copula log-likelihood
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    lpdf = special.log1p(theta) - (1+theta)*(np.log(u0)+ np.log(u1)) - (2+1/theta)*clayton_logAfunc(u, theta)
    return np.sum(lpdf)

def clayton_lpdf_grad(u, theta):
    """
    Clayton copula gradient w.r.t copula parameter theta

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Clayton copula lpdf gradient
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    dlogA = clayton_logA_grad(u, theta)
    lpdf_grad = 1/(1+theta) - (np.log(u0) + np.log(u1)) - (2 + 1/theta)*dlogA + clayton_logAfunc(u, theta)/np.square(theta)
    return lpdf_grad
    
def clayton_link(par):
    """
    :param par: current parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: link function of copula parameter
    """
    return np.log(par)
    
def clayton_inv_link(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: inverse link function of transformed copula parameter
    """
    return np.exp(tpar)
    
def clayton_inv_link_grad(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    return: gradient of inverse link function w.r.t transformed copula parameter
    """
    return np.exp(tpar)

def clayton_gradient(u: np.ndarray, predt: np.ndarray) -> np.ndarray:
    '''Compute the gradient of the nll'''
    tau = special.expit(predt)
    theta = clayton.clayton_tau2par(tau)
    return -clayton_lpdf_grad(u, theta)*(2*tau/(1-tau))
        
def clayton_copula_nll_obj(u: np.ndarray, predt: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''Clayton copula negative log-likelihood objective'''
    grad = clayton_gradient(u, predt)
    hess = grad**2 #BHHH hessian substitute
    return grad, hess

def clayton_copula_nll_metric(u: np.ndarray, predt: np.ndarray) -> Tuple[str, float]:
    ''' Clayton copula negative log-likelihood metric'''
    tau = special.expit(predt)
    theta = clayton.clayton_tau2par(tau)
    log_likelihood = clayton_lpdf(u, theta)
    return float(-log_likelihood)

In [ ]:
def gumbel_lpdf(u, theta):
    """
    Gumbel copula log-likelihood evaluation

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param rho: current copula parameter estimates (n_obs, 1)
        
    :return: Gumbel copula log-likelihood
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    t = (-np.log(u0))**theta
    s = (-np.log(u1))**theta
    a = (t + s)**(1/theta)

    return np.sum(-a - np.log(u0*u1) + (-2+2/theta)*np.log(t+s) + (theta - 1)*(np.log(np.log(u0)*np.log(u1))) + np.log(1+(theta-1)/a))
    
def gumbel_lpdf_grad(u, theta):
    """
    Gumbel copula gradient w.r.t dependence parameter theta

    :param u: bivariate pseudo-observation vector (n_obs, 2)
    :param theta: current dependence parameter estimates (n_obs, 1)
        
    :return: Gumbel copula lpdf gradient
    """
    u0 = u[:, 0].reshape(-1,1)
    u1 = u[:, 1].reshape(-1,1)
    t = (-np.log(u0))**theta
    s = (-np.log(u1))**theta
    a = (t + s)**(1/theta)
    b = np.log(t+s)/np.square(theta)
    d = (t*np.log(-np.log(u0)) + s*np.log(-np.log(u1)))/(t+s)

    return -a*(-b + d/theta) + (-2*b + (-2+2/theta)*d) + np.log(np.log(u0)*np.log(u1)) + a/(a + (theta - 1))*(1/a + ((theta-1)/a)*(b - d/theta))
    
def gumbel_link(par):
    """
    :param par: current parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: link function of copula parameter
    """
    return np.log(par-1)
    
def gumbel_inv_link(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    :return: inverse link function of transformed copula parameter
    """
    return np.exp(tpar)+1
    
def gumbel_inv_link_grad(tpar):
    """
    :param tpar: current (transformed) parameter estimates (n_obs, n_param) (e.g. for a bivariate copula, n_param = 1)

    return: gradient of inverse link function w.r.t transformed copula parameter
    """
    return np.exp(tpar)

def gumbel_gradient(u: np.ndarray, predt: np.ndarray) -> np.ndarray:
    '''Compute the gradient of the nll'''
    tau = special.expit(predt)
    theta = gumbel.gumbel_tau2par(tau)
    return -gumbel_lpdf_grad(u, theta)*(2*tau/(1-tau))
        
def gumbel_copula_nll_obj(u: np.ndarray, predt: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    '''Gumbel copula negative log-likelihood objective'''
    grad = gumbel_gradient(u, predt)
    hess = grad**2 #BHHH hessian substitute
    return grad, hess

def gumbel_copula_nll_metric(u: np.ndarray, predt: np.ndarray) -> Tuple[str, float]:
    ''' Gumbel copula negative log-likelihood metric'''
    tau = special.expit(predt)
    theta = gumbel.gumbel_tau2par(tau)
    log_likelihood = gumbel_lpdf(u, theta)
    return float(-log_likelihood)

In [10]:
from sklearn.metrics import make_scorer
from xgboost import XGBRegressor

gauss_scorer = make_scorer(gaussian_copula_nll_metric, greater_is_better=False)

gauss_model = XGBRegressor(
    objective=gaussian_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

clayton_scorer = make_scorer(clayton_copula_nll_metric, greater_is_better=False)

clayton_model = XGBRegressor(
    objective=clayton_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

gumbel_scorer = make_scorer(gumbel_copula_nll_metric, greater_is_better=False)

gumbel_model = XGBRegressor(
    objective=gumbel_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV

param_grid = {
    'max_depth': [1, 2, 3, 5],
    'eta':[0.01, 0.1, 0.3],
    'lambda': [0.0, 0.1, 0.2]
}

grid_search_gauss=GridSearchCV(
    estimator=gauss_model,
    param_grid=param_grid,
    scoring=gauss_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_clayton=GridSearchCV(
    estimator=clayton_model,
    param_grid=param_grid,
    scoring=clayton_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_gumbel=GridSearchCV(
    estimator=gumbel_model,
    param_grid=param_grid,
    scoring=gumbel_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

In [18]:
y_gauss1 = clip_copula_samples(sample_gaussian_copula(gauss.gauss_tau2par(tau1), 0), eps=1e-7)
x_train_gauss1, _, y_train_gauss1, _ = train_test_split(x, y_gauss1, test_size=1-ratio, random_state=42)

grid_search_gauss.fit(x_train_gauss1, y_train_gauss1)

print(f"Best parameters (gauss1): {grid_search_gauss.best_params_}")
print(f"Best score (gauss1): {grid_search_gauss.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gauss1): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 2}
Best score (gauss1): 170.69311336052465


In [19]:
y_clayton1 = clip_copula_samples(sample_clayton_copula(clayton.clayton_tau2par(tau1), 0), eps=1e-7)
x_train_clayton1, _, y_train_clayton1, _ = train_test_split(x, y_clayton1, test_size=1-ratio, random_state=42)

grid_search_clayton.fit(x_train_clayton1, y_train_clayton1)

print(f"Best parameters (clayton1): {grid_search_clayton.best_params_}")
print(f"Best score (clayton1): {grid_search_clayton.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (clayton1): {'eta': 0.01, 'lambda': 0.1, 'max_depth': 1}
Best score (clayton1): 216.85828214949612


In [20]:
y_gumbel1 = clip_copula_samples(sample_gumbel_copula(gumbel.gumbel_tau2par(tau1), 0), eps=1e-7)
x_train_gumbel1, _, y_train_gumbel1, _ = train_test_split(x, y_gumbel1, test_size=1-ratio, random_state=42)

grid_search_gumbel.fit(x_train_gumbel1, y_train_gumbel1)

print(f"Best parameters (gumbel1): {grid_search_gumbel.best_params_}")
print(f"Best score (gumbel1): {grid_search_gumbel.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gumbel1): {'eta': 0.1, 'lambda': 0.2, 'max_depth': 1}
Best score (gumbel1): 203.1420971381602


In [21]:
y_gauss2 = clip_copula_samples(sample_gaussian_copula(gauss.gauss_tau2par(tau2), 0), eps=1e-7)
x_train_gauss2, _, y_train_gauss2, _ = train_test_split(x, y_gauss2, test_size=1-ratio, random_state=42)

grid_search_gauss.fit(x_train_gauss2, y_train_gauss2)

print(f"Best parameters (gauss2): {grid_search_gauss.best_params_}")
print(f"Best score (gauss2): {grid_search_gauss.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gauss2): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 2}
Best score (gauss2): 344.9889402595797


In [22]:
y_clayton2 = clip_copula_samples(sample_clayton_copula(clayton.clayton_tau2par(tau2), 0), eps=1e-7)
x_train_clayton2, _, y_train_clayton2, _ = train_test_split(x, y_clayton2, test_size=1-ratio, random_state=42)

grid_search_clayton.fit(x_train_clayton2, y_train_clayton2)

print(f"Best parameters (clayton2): {grid_search_clayton.best_params_}")
print(f"Best score (clayton2): {grid_search_clayton.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (clayton2): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (clayton2): 411.2764274334836


In [23]:
y_gumbel2 = clip_copula_samples(sample_gumbel_copula(gumbel.gumbel_tau2par(tau2), 0), eps=1e-7)
x_train_gumbel2, _, y_train_gumbel2, _ = train_test_split(x, y_gumbel2, test_size=1-ratio, random_state=42)

grid_search_gumbel.fit(x_train_gumbel2, y_train_gumbel2)

print(f"Best parameters (gumbel2): {grid_search_gumbel.best_params_}")
print(f"Best score (gumbel2): {grid_search_gumbel.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gumbel2): {'eta': 0.01, 'lambda': 0.2, 'max_depth': 2}
Best score (gumbel2): 392.5888153794538
